
# 🎨 Advanced Paint-by-Numbers (SAM + U-Net Style Segmentation)

## Overview
This notebook upgrades basic KMeans to **Segmentation Models**:
- Meta SAM (Segment Anything) for region detection
- Optional U-Net style refinement
- Cleaner boundaries for paint-by-number output
- Number overlay + color legend
- Gradio UI

---


In [21]:

!pip install torch torchvision pillow numpy opencv-python gradio segment-anything scikit-learn


In [12]:

import numpy as np
import cv2
from PIL import Image, ImageDraw
import gradio as gr
from sklearn.cluster import KMeans

# SAM imports
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator


## 🧠 Load SAM Model

In [13]:

def load_sam_model():
    """
    Loads Segment Anything Model (SAM)

    Returns:
        mask_generator: SAM automatic mask generator
    """
    sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b.pth")
    mask_generator = SamAutomaticMaskGenerator(sam)
    return mask_generator


## ⚙️ Core Functions

In [14]:

def generate_masks(image, mask_generator):
    """
    Generate segmentation masks using SAM

    Args:
        image (np.array): Input image
        mask_generator: SAM model

    Returns:
        list: segmentation masks
    """
    masks = mask_generator.generate(image)
    return masks


In [15]:

def extract_regions(image, masks):
    """
    Extract regions from masks

    Returns:
        list of region pixel arrays
    """
    regions = []
    for m in masks:
        mask = m['segmentation']
        region = image * mask[..., None]
        regions.append(region)
    return regions


In [16]:

def assign_colors(image, masks, k=8):
    """
    Assign dominant colors to each region

    Returns:
        segmented image, palette
    """
    output = np.zeros_like(image)
    palette = []

    for i, m in enumerate(masks):
        mask = m['segmentation']
        pixels = image[mask]

        if len(pixels) == 0:
            continue

        kmeans = KMeans(n_clusters=1, random_state=0).fit(pixels)
        color = kmeans.cluster_centers_[0].astype(int)
        palette.append(color)

        output[mask] = color

    return output, palette


In [17]:

def overlay_numbers(image, masks):
    """
    Overlay region numbers

    Returns:
        PIL Image
    """
    img = Image.fromarray(image.astype('uint8'))
    draw = ImageDraw.Draw(img)

    for i, m in enumerate(masks):
        y, x = np.where(m['segmentation'])
        if len(x) > 0:
            draw.text((int(x.mean()), int(y.mean())), str(i+1), fill=(0,0,0))

    return img


In [18]:

def generate_legend(palette):
    """
    Generate color legend

    Returns:
        str
    """
    return "\n".join([f"{i+1}: RGB{tuple(map(int,c))}" for i,c in enumerate(palette)])


## 🔄 Pipeline

In [19]:

def pipeline(image):
    """
    Full segmentation-based paint-by-number pipeline
    """
    image_np = np.array(image)

    mask_generator = load_sam_model()
    masks = generate_masks(image_np, mask_generator)

    segmented, palette = assign_colors(image_np, masks)
    numbered = overlay_numbers(segmented, masks)
    legend = generate_legend(palette)

    return numbered, legend


## 🎨 Gradio UI

In [23]:

def launch_app():
    """
    Launch Gradio interface
    """
    try:
      with gr.Blocks(theme=gr.themes.Soft()) as demo:
          gr.Markdown("# 🎨 SAM-based Paint by Numbers")

          api_key = gr.Textbox(label="Groq API Key (optional)", type="password")
          image = gr.Image(type="pil")

          output = gr.Image(label="Output Image")
          legend = gr.Textbox(label="Color Legend")

          btn = gr.Button("Generate")

          btn.click(pipeline, inputs=[image], outputs=[output, legend])

      demo.launch(debug=True)
      print("Gradio app launched successfully.")
    except Exception as e:
      print(f"Error launching Gradio app: {e}")


In [24]:
launch_app()

/tmp/ipykernel_25942/893205093.py:6: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://988a1d607f7baa41a4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio app launched successfully.
